# Phase 3a - Pose Format Exploration

Before writing `_load_poses()`, confirm the facts nobody has ever checked: real-camera
poses are "one `.xlsx` per trajectory" per an early folder listing, but the column
layout, rotation representation, translation units, and row-to-frame alignment are all
unconfirmed. UnityCam's `Poses/` format (even the file extension) is unconfirmed too.

Same probe-first pattern that got Phase 1's dataset loader and Phase 2's DarkIR loading
API right before code was written against guesses -- see `phase2a_explore` for the
precedent. **GPU is off** -- this is inspection only, no training, no model.

Cells are defensive (try/except + prints), not hard-asserting -- the goal is gathering
facts to write into `PROGRESS.md`, then `_load_poses()` gets implemented against those
confirmed facts.

## 0. Setup: clone repo, resolve dataset root, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q pandas openpyxl

import os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

DATA_ROOT = find_endoslam_root()
print("resolved dataset root:", DATA_ROOT)
assert DATA_ROOT is not None, "could not resolve EndoSLAM dataset root -- check mount path"


## 1. Real-camera pose file: locate and count

In [ ]:
import glob

def real_cam_traj_dir(root, cam, organ_roman, traj_n):
    return os.path.join(root, "Cameras", cam, f"Stomach-{organ_roman}", f"TumorfreeTrajectory_{traj_n}")

traj_dir = real_cam_traj_dir(DATA_ROOT, "HighCam", "I", 1)
print("trajectory dir:", traj_dir)
print("contents:", sorted(os.listdir(traj_dir)) if os.path.isdir(traj_dir) else "NOT FOUND")

pose_files = sorted(glob.glob(os.path.join(traj_dir, "Poses", "*")))
print(f"\n{len(pose_files)} file(s) under Poses/:")
for f in pose_files:
    print(" ", f)
# Confirms (or refutes) the "one .xlsx per trajectory" claim from the original folder
# listing -- if this prints more than one file, poses may be per-frame instead.


## 2. Read the xlsx: sheets, columns, dtypes, row count

In [ ]:
import pandas as pd

def inspect_pose_xlsx(path):
    sheets = pd.read_excel(path, sheet_name=None)
    print("file:", path)
    print("sheet names:", list(sheets.keys()))
    for name, df in sheets.items():
        print(f"\n--- sheet '{name}' ---")
        print("columns:", df.columns.tolist())
        print("dtypes:\n", df.dtypes)
        print("row count:", len(df))
        print("head:\n", df.head())
        print("describe:\n", df.describe())
    return sheets

pose_sheets_1 = None
if pose_files:
    pose_sheets_1 = inspect_pose_xlsx(pose_files[0])
else:
    print("no pose file found in cell above -- nothing to inspect")


## 3. Compare pose row count vs. frame count; look for an alignment column

In [ ]:
frame_paths = sorted(
    glob.glob(os.path.join(traj_dir, "Frames", "*.jpg"))
    + glob.glob(os.path.join(traj_dir, "Frames", "*.png"))
)
print("frame count:", len(frame_paths))

if pose_sheets_1:
    first_sheet = next(iter(pose_sheets_1.values()))
    print("pose row count:", len(first_sheet))
    print("row_count == frame_count:", len(first_sheet) == len(frame_paths))
    # look for anything that could support name/index/timestamp-based matching
    # instead of assumed positional order
    candidate_cols = [c for c in first_sheet.columns
                       if any(k in str(c).lower() for k in ("frame", "name", "index", "time", "id"))]
    print("candidate alignment columns:", candidate_cols)
    if candidate_cols:
        print(first_sheet[candidate_cols].head(10))


## 4. Sniff rotation representation from column names

In [ ]:
import numpy as np

if pose_sheets_1:
    cols_lower = [str(c).lower() for c in first_sheet.columns]
    quat_cols = [c for c in cols_lower if c in ("qx", "qy", "qz", "qw")]
    euler_cols = [c for c in cols_lower if c in ("roll", "pitch", "yaw", "rx", "ry", "rz")]
    matrix_cols = [c for c in cols_lower if c.startswith("r") and len(c) == 3 and c[1:].isdigit()]
    print("quaternion-like columns:", quat_cols)
    print("euler-like columns:", euler_cols)
    print("matrix-like columns:", matrix_cols)
    print("ALL columns for manual inspection:", first_sheet.columns.tolist())

    if len(quat_cols) == 4:
        q = first_sheet[[c for c in first_sheet.columns if str(c).lower() in quat_cols]].to_numpy()
        norms = np.linalg.norm(q, axis=1)
        print("quaternion norm stats: min", norms.min(), "max", norms.max(), "mean", norms.mean())
        print("(should be ~1.0 if these are unit quaternions)")


## 5. Sniff translation units

In [ ]:
if pose_sheets_1:
    trans_cols = [c for c in first_sheet.columns if str(c).lower() in ("x", "y", "z", "tx", "ty", "tz")]
    print("translation-like columns:", trans_cols)
    if trans_cols:
        t = first_sheet[trans_cols].to_numpy()
        mag = np.linalg.norm(t, axis=1)
        print("translation magnitude: min", mag.min(), "max", mag.max(), "mean", mag.mean())
        print("compare against config.yaml fusion.depth_trunc: 0.15 (meters, endoscope-scale)")
        print("if magnitudes are ~0.01-0.2 -> likely meters; ~10-200 -> likely mm; ~1-20 -> likely cm")


## 6. Repeat for a different real-camera trajectory + organ (schema stability check)

In [ ]:
def summarize(cam, organ_roman, traj_n):
    d = real_cam_traj_dir(DATA_ROOT, cam, organ_roman, traj_n)
    pf = sorted(glob.glob(os.path.join(d, "Poses", "*")))
    fr = sorted(glob.glob(os.path.join(d, "Frames", "*.jpg")) + glob.glob(os.path.join(d, "Frames", "*.png")))
    print(f"=== {cam}/Stomach-{organ_roman}/TumorfreeTrajectory_{traj_n} ===")
    print("pose files:", pf)
    print("frame count:", len(fr))
    if pf:
        sheets = pd.read_excel(pf[0], sheet_name=None)
        sheet = next(iter(sheets.values()))
        print("columns:", sheet.columns.tolist())
        print("row count:", len(sheet), "vs frame count:", len(fr))
    return pf, fr

_ = summarize("LowCam", "II", 2)
print()
_ = summarize("HighCam", "III", 3)


## 7. UnityCam Poses/ -- format is completely unknown

In [ ]:
unity_organ_dir = os.path.join(DATA_ROOT, "UnityCam", "Stomach")
unity_poses_dir = os.path.join(unity_organ_dir, "Poses")
print("UnityCam Poses/ dir:", unity_poses_dir)
unity_pose_entries = sorted(os.listdir(unity_poses_dir)) if os.path.isdir(unity_poses_dir) else []
print(f"{len(unity_pose_entries)} entries:")
for e in unity_pose_entries[:20]:
    print(" ", e)
if len(unity_pose_entries) > 20:
    print(f"  ... and {len(unity_pose_entries) - 20} more")

# Inspect the first entry's raw content before assuming a parser -- extension alone
# may not tell the whole story (could be a single manifest file, or per-frame files).
if unity_pose_entries:
    first_entry_path = os.path.join(unity_poses_dir, unity_pose_entries[0])
    print("\nfirst entry:", first_entry_path)
    print("is file:", os.path.isfile(first_entry_path), "is dir:", os.path.isdir(first_entry_path))
    if os.path.isfile(first_entry_path):
        with open(first_entry_path, "rb") as f:
            raw = f.read(500)
        print("first 500 bytes (raw):", raw)


## 8. Parse UnityCam poses per whatever format cell 7 reveals

In [ ]:
unity_frame_paths = sorted(
    glob.glob(os.path.join(unity_organ_dir, "Frames", "*.png"))
    + glob.glob(os.path.join(unity_organ_dir, "Frames", "*.jpg"))
)
unity_depth_paths = sorted(glob.glob(os.path.join(unity_organ_dir, "Pixelwise Depths", "*")))
print("UnityCam frame count:", len(unity_frame_paths))
print("UnityCam depth count:", len(unity_depth_paths))
print("UnityCam pose entry count:", len(unity_pose_entries))

# Try the obvious cases based on what cell 7 found. Adjust this cell once the real
# format is visible above -- don't guess further blind.
if len(unity_pose_entries) == 1:
    entry_path = os.path.join(unity_poses_dir, unity_pose_entries[0])
    ext = os.path.splitext(entry_path)[1].lower()
    print("single-file case, extension:", ext)
    if ext in (".xlsx", ".xls"):
        sheets = pd.read_excel(entry_path, sheet_name=None)
        sheet = next(iter(sheets.values()))
        print("columns:", sheet.columns.tolist())
        print("row count:", len(sheet), "vs frame count:", len(unity_frame_paths))
    elif ext in (".csv", ".txt"):
        df = pd.read_csv(entry_path)
        print("columns:", df.columns.tolist())
        print("row count:", len(df), "vs frame count:", len(unity_frame_paths))
    else:
        print("unrecognized single-file extension -- inspect raw bytes from cell 7 manually")
elif len(unity_pose_entries) > 1:
    print("multi-file case (likely per-frame) -- row/file count comparison:")
    print("pose entries:", len(unity_pose_entries), "vs frames:", len(unity_frame_paths))
else:
    print("no entries found under Poses/ -- check unity_poses_dir path above")


## Done

Collect the findings from every section above -- real-camera `.xlsx` column names/dtypes,
row-vs-frame-count match (and any alignment column found), rotation representation,
translation units, schema stability across trajectories/organs, and UnityCam's actual
pose format -- and write them into `PROGRESS.md` as a new "Pose format -- confirmed
facts" subsection (matching the existing "DarkIR loading -- confirmed facts" style).
Then implement `_load_real_camera_poses()` / `_load_unitycam_poses()` in
`src/data/endoslam_dataset.py` against these confirmed facts -- don't guess further
locally.